# 07 · As a Tool — 01 a function as a tool

**Runs end to end with no API key.** Everything below is plain Python — `inspect`, `json`, `re`. The one optional cell that would use a real model reports through `nbio.show_environment()` and runs a deterministic stand-in chooser instead when no key is loaded.

A tool is not a special kind of object. It is an ordinary function plus three pieces of text and structure the model is shown before it is asked anything: a **name**, a **description**, and a **JSON schema for the arguments**. That is the entire contract. This notebook builds the thing that produces those three pieces from a normal Python function, prints the resulting JSON so you can read exactly what the model receives, and then proves the part everybody underestimates: the docstring is not documentation for humans. It is the input the model uses to decide whether to call the function at all. A function with a vague description is a function that never gets called.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `tool_spec` | Turns an ordinary function into the name/description/JSON-schema record a model is shown. | `tool_spec(order_history)` -> `{"name": ..., "description": ..., "parameters": {...}}` |
| `@tool` | Registers a function in a dict of callable tools, returning the function unchanged. | `@tool` above `def order_history(...)` -> `REGISTRY["order_history"]` |
| `_arg_docs` | Reads per-argument descriptions out of a docstring's `Args:` block into the schema. | `_arg_docs(doc)` -> `{"topic": "One of returns, shipping, or warranty."}` |
| `choose_tool` | A deterministic stand-in for a model picking a tool: scores each spec's text against the question. | `choose_tool(question, specs)` -> `("order_history", [scores...])` |
| The docstring swap | Sharpens one description, changes no code, and flips which tool is picked. | vague -> wrong tool; sharp -> right tool |

## Step 1 — bootstrap the repo path

Jupyter starts this kernel with the notebook's own directory as `cwd`, not the repo root, so `nbio.py` has to be located and put on `sys.path` before it can be imported. This notebook sits three levels below the root, which the six-step walk-up covers.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

## Step 2 — say which path this run is on

One optional cell near the end calls a real model. Print which keys are loaded now, so there is never any doubt about whether the tool choice you read further down came from a model or from the deterministic stand-in.

In [ ]:
nbio.show_environment()

## Step 3 — the three things a model is shown

`tool_spec` reads a function's signature, its type hints, and its docstring, and emits the JSON record a provider's tool/function-calling API expects: a name, a description, and a JSON Schema object for the arguments. Nothing here is provider-specific magic — a framework that "registers tools for you" is doing this, and roughly only this.

Two details worth naming, because they are where real tool specs go wrong:

- **`required` is derived from the signature.** A parameter with a default is optional; one without is required. Get this wrong and the model omits an argument the function cannot run without.
- **Per-argument descriptions come from the `Args:` block.** The model gets one sentence per argument and nothing else — no access to the body, no way to guess what the units are.

In [ ]:
import inspect
import json
import re
import typing

# The only Python types this mapper handles; anything else falls through to
# "string", which is the honest default when a schema cannot express the type.
_JSON_TYPES = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean",
    list: "array",
    dict: "object",
}

_ARG_HEADINGS = {"args", "arguments", "parameters"}


def _arg_docs(doc: str) -> dict[str, str]:
    """Read `name: description` pairs out of a docstring's Args: block."""
    out: dict[str, str] = {}
    in_args = False
    for line in (doc or "").splitlines():
        stripped = line.strip()
        if stripped.lower().rstrip(":") in _ARG_HEADINGS and stripped.endswith(":"):
            in_args = True
            continue
        if not in_args:
            continue
        if not stripped:
            continue
        if ":" not in stripped:
            break
        name, _, desc = stripped.partition(":")
        out[name.strip().split("(")[0].strip()] = desc.strip()
    return out


def _description(doc: str) -> str:
    """Everything above the Args: block, collapsed to one paragraph."""
    head: list[str] = []
    for line in (doc or "").splitlines():
        if line.strip().lower().rstrip(":") in _ARG_HEADINGS:
            break
        head.append(line.strip())
    return " ".join(part for part in head if part).strip()


def tool_spec(fn) -> dict:
    """The name, description and argument schema a model is shown for one function."""
    doc = inspect.getdoc(fn) or ""
    arg_docs = _arg_docs(doc)
    hints = typing.get_type_hints(fn)

    properties: dict[str, dict] = {}
    required: list[str] = []
    for name, param in inspect.signature(fn).parameters.items():
        prop: dict = {"type": _JSON_TYPES.get(hints.get(name, str), "string")}
        if name in arg_docs:
            prop["description"] = arg_docs[name]
        if param.default is inspect.Parameter.empty:
            required.append(name)
        else:
            prop["default"] = param.default
        properties[name] = prop

    return {
        "name": fn.__name__,
        "description": _description(doc),
        "parameters": {"type": "object", "properties": properties, "required": required},
    }

## Step 4 — a registry, in six lines

`@tool` puts the function and its spec in a dict and returns the function **unchanged**. That is deliberate: a tool is still an ordinary function that ordinary Python can call directly. The registry is the lookup table an agent loop uses later to turn a name the model returned back into something callable.

In [ ]:
REGISTRY: dict[str, dict] = {}


def tool(fn):
    """Register a function as a callable tool. Returns it unchanged."""
    REGISTRY[fn.__name__] = {"fn": fn, "spec": tool_spec(fn)}
    return fn

## Step 5 — register one real function and print the JSON the model receives

`order_history` below is a normal function over a tiny synthetic order list — invented for this notebook, no real customer data. Read the printed record as the model reads it: this is *all* it gets. It does not see the body, the data, or this markdown.

In [ ]:
ORDERS = [
    {"order_id": "A-1001", "item": "wool scarf", "shipped": "2026-08-02", "price_usd": 38.0},
    {"order_id": "A-1042", "item": "ceramic mug", "shipped": "2026-08-19", "price_usd": 14.5},
    {"order_id": "A-1108", "item": "desk lamp", "shipped": "2026-09-04", "price_usd": 62.0},
]


@tool
def order_history(customer_id: str, limit: int = 3) -> list[dict]:
    """Look up a customer's past purchases, including each item bought and the date each one shipped.

    Args:
        customer_id: The account id whose purchases should be listed.
        limit: How many purchases to return, most recent first.
    """
    return sorted(ORDERS, key=lambda o: o["shipped"], reverse=True)[:limit]


print(json.dumps(REGISTRY["order_history"]["spec"], indent=2))

## Step 6 — check the spec against the signature, not against the prose above

Three claims worth a real assertion: the required list came from the signature (so `limit`, which has a default, is *not* required), the argument types came from the hints, and the decorator handed back a function that plain Python can still call.

In [ ]:
spec = REGISTRY["order_history"]["spec"]

assert spec["name"] == "order_history"
assert spec["parameters"]["required"] == ["customer_id"], spec["parameters"]["required"]
assert spec["parameters"]["properties"]["limit"]["type"] == "integer"
assert spec["parameters"]["properties"]["limit"]["default"] == 3
assert "account id" in spec["parameters"]["properties"]["customer_id"]["description"]

direct = order_history("cust-7", limit=2)
assert REGISTRY["order_history"]["fn"] is order_history

print(f"required from the signature : {spec['parameters']['required']}")
print(f"optional (has a default)    : {[k for k, v in spec['parameters']['properties'].items() if 'default' in v]}")
print(f"still an ordinary function  : order_history('cust-7', limit=2) -> {len(direct)} rows")
nbio.table(
    [(o["order_id"], o["item"], o["shipped"]) for o in direct],
    headers=("order_id", "item", "shipped"),
)

## Step 7 — a second function, described badly on purpose

`company_handbook` is the tool that *should* answer a question about a return window. Its docstring here is the kind of description that gets written for a human who already knows what the function is for: **"Gets information from the handbook."** Every word of that is true and none of it is useful to a model deciding between two tools.

In [ ]:
HANDBOOK = {
    "returns": (
        "Items may be returned within 30 days of the shipping date for a full refund, "
        "provided they are unused and in original packaging."
    ),
    "shipping": "Standard shipping takes 3-5 business days; expedited shipping takes 1-2.",
    "warranty": "Electronics carry a one-year limited warranty against manufacturing defects.",
}


@tool
def company_handbook(topic: str) -> str:
    """Gets information from the handbook."""
    return HANDBOOK.get(topic.lower().strip(), "No handbook entry for that topic.")


print(json.dumps(REGISTRY["company_handbook"]["spec"], indent=2))

## Step 8 — the stand-in chooser

With no key loaded, something still has to make a choice, and it has to make it from the same input a model would get. This chooser scores the overlap between the question's words and each spec's **name plus description** — nothing else. It is crude on purpose (exact token match, no stemming, a short stopword list), and it is not a model. What it shares with a model is the only thing that matters here: the description is the evidence it decides on.

In [ ]:
STOPWORDS = {
    "a", "an", "and", "are", "at", "be", "by", "can", "did", "do", "does", "each",
    "for", "from", "has", "have", "how", "i", "in", "is", "it", "many", "me", "my",
    "of", "on", "one", "our", "that", "the", "this", "to", "up", "what", "which",
    "who", "with", "you", "your",
}


def _tokens(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z]+", (text or "").lower()) if w not in STOPWORDS}


def choose_tool(question: str, specs: list[dict]) -> tuple[str, list[tuple[str, int]]]:
    """Pick a tool by word overlap with its name and description. A stand-in for a model."""
    q = _tokens(question)
    scored = [(s["name"], len(_tokens(s["name"] + " " + s["description"]) & q)) for s in specs]
    # Ties break alphabetically so the result is reproducible run to run.
    scored.sort(key=lambda pair: (-pair[1], pair[0]))
    return scored[0][0], scored

## Step 9 — the wrong tool gets picked

The question is squarely a handbook question. Watch which tool wins, and why: `order_history` matched on the single word *item*, and `company_handbook`'s description matched nothing at all, because it says nothing at all.

In [ ]:
QUESTION = "How many days do I have to return an item for a refund?"
specs = [entry["spec"] for entry in REGISTRY.values()]

picked_vague, scores_vague = choose_tool(QUESTION, specs)
nbio.table(list(scores_vague), headers=("tool", "overlap with the question"))
print()
print(f"question : {QUESTION}")
print(f"picked   : {picked_vague}   <- wrong: the answer is in the handbook, not the order list")

assert picked_vague == "order_history", "expected the vague description to lose the question it should have won"

## Step 10 — change the description, change nothing else

The next cell rewrites `company_handbook.__doc__` and re-registers it. It does not touch the body, the signature, or the data. The proof that nothing executable changed is in the assertions: the same code object, and the same return value for the same argument.

In [ ]:
code_before = company_handbook.__code__
answer_before = company_handbook("returns")

company_handbook.__doc__ = """Look up the official refund and return policy: how many days a customer has to return an item, plus shipping times and warranty terms.

Args:
    topic: One of "returns", "shipping", or "warranty".
"""
tool(company_handbook)  # re-register, so the registry holds the new spec

assert company_handbook.__code__ is code_before, "the function body must be untouched"
assert company_handbook("returns") == answer_before, "behaviour must be identical"

print(json.dumps(REGISTRY["company_handbook"]["spec"], indent=2))

## Step 11 — the same question, the same code, the right tool

Nothing about what this function *does* changed between Step 9 and here. Only what it *says about itself* changed, and that is the whole difference between a tool that gets called and a tool that sits there unused while something worse answers in its place.

In [ ]:
specs = [entry["spec"] for entry in REGISTRY.values()]
picked_sharp, scores_sharp = choose_tool(QUESTION, specs)
nbio.table(list(scores_sharp), headers=("tool", "overlap with the question"))
print()
print(f"question : {QUESTION}")
print(f"picked   : {picked_sharp}")
print(f"result   : {REGISTRY[picked_sharp]['fn']('returns')}")

assert picked_sharp == "company_handbook", "the sharpened description should now win"
assert picked_vague != picked_sharp, "the description swap must be what flipped the choice"
print()
print(f"vague description -> {picked_vague}")
print(f"sharp description -> {picked_sharp}")

## Step 12 — the same two specs, sent to a real model — only if a key is loaded

The specs printed above are already in the shape a provider's tool-calling API accepts. If `GROQ_API_KEY` or `OPENAI_API_KEY` is loaded, this cell hands both of them to a real model, asks the same question, and prints the tool the model asked to call — under a spend ceiling, because a tool-choice loop is exactly the shape that runs away. With no key it says so and stops; the deterministic result above already made the point without spending anything.

In [ ]:
import os


def _client():
    if os.environ.get("GROQ_API_KEY"):
        from groq import Groq

        return "groq", Groq(api_key=os.environ["GROQ_API_KEY"]), "llama-3.3-70b-versatile"
    if os.environ.get("OPENAI_API_KEY"):
        from openai import OpenAI

        return "openai", OpenAI(api_key=os.environ["OPENAI_API_KEY"]), "gpt-4o-mini"
    return None, None, None


provider, client, model_id = _client()

with nbio.cost_meter(budget_usd=0.50) as meter:
    if provider is None:
        print(
            "No GROQ_API_KEY or OPENAI_API_KEY loaded (see the environment readout in Step 2) -- "
            "not set, skipping. Running the deterministic stand-in instead: the chooser in Steps "
            "9 and 11 produced the tool picks above, and it is not a model."
        )
    else:
        print(f"asking a real model to choose, provider={provider!r} model={model_id!r}")
        resp = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": QUESTION}],
            tools=[{"type": "function", "function": s} for s in specs],
            temperature=0.0,
        )
        calls = resp.choices[0].message.tool_calls or []
        usage = getattr(resp, "usage", None)
        meter.record(
            model_id,
            getattr(usage, "prompt_tokens", 0) if usage else 0,
            getattr(usage, "completion_tokens", 0) if usage else 0,
        )
        if calls:
            print(f"model chose : {calls[0].function.name}")
            print(f"arguments   : {calls[0].function.arguments}")
        else:
            print("model chose : (no tool call -- it answered directly)")

print()
print(meter.report())

## What did not come across

- **A real model is not a bag of words.** The stand-in chooser matches tokens; a model reads a description semantically and would quite possibly have picked `company_handbook` even from a mediocre one. The effect is real, but this chooser makes it look sharper and more mechanical than it is. What is not exaggerated: the description is the highest-leverage prose in an agent, and it is the only part of your function the model ever reads.
- **Argument-schema validation.** `tool_spec` describes the arguments; nothing here checks that what came back matches. A real system validates arguments against the schema before calling anything, and has to decide what to do when they don't match — that is `04-tool-failure.ipynb`.
- **Richer schema.** No enums, no nested objects, no typed array items, no `additionalProperties: false`. `topic` is typed `string` here when it is really an enum of three values; a real spec would say so and delete a whole class of bad calls.
- **The rest of the loop.** Choosing is not calling. Getting from a chosen name back to a result, and from a result back to an answer, is `03-a-second-tool.ipynb`.

Next: `02-pipeline-as-tool.ipynb` — six stages of retrieval, collapsed into one function with a schema like the one printed above.